In [1]:
import torch
torch.manual_seed(789)


# Attention v1

In [2]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec



# Attention v2

In [3]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec


# create fake inputs

In [20]:

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [21]:
inputs.shape

torch.Size([6, 3])

In [30]:
d_in = inputs.shape[1] # the input embedding size, d=3
d_out=2

print("input embedding dim:", d_in)
print("output embedding dim:", d_out)

input embedding dim: 3
output embedding dim: 2


# create an instance of self attention v2

In [23]:
sa_v2 = SelfAttention_v2(d_in, d_out)
#print(sa_v2(inputs))

In [24]:
# get state dict from self attention v2 class

In [25]:
sa_sd = sa_v2.state_dict()

In [26]:
print(sa_sd.keys())

odict_keys(['W_query.weight', 'W_key.weight', 'W_value.weight'])


# get the weights of each linear layer

In [27]:
W_query_weight = sa_v2.state_dict()['W_query.weight']
W_key_weight = sa_v2.state_dict()['W_key.weight']
W_value_weight = sa_v2.state_dict()['W_value.weight']

In [28]:
W_query_weight.shape

torch.Size([2, 3])

In [ ]:
# the matrix has the shape [d_in, d_out]
# it must be transposed

In [31]:
torch.transpose(W_query_weight, 0, 1)

tensor([[ 0.3847,  0.4892],
        [-0.4937, -0.2288],
        [-0.3170, -0.1765]])

# create instance of self attention v1

In [32]:
sa_v1 = SelfAttention_v1(d_in, d_out)


In [33]:
sa_v1.W_query = torch.nn.Parameter(torch.transpose(W_query_weight, 0, 1))
sa_v1.W_key = torch.nn.Parameter(torch.transpose(W_key_weight, 0, 1))
sa_v1.W_value = torch.nn.Parameter(torch.transpose(W_value_weight, 0, 1))

In [34]:
sa_v1.state_dict()

OrderedDict([('W_query',
              tensor([[ 0.3847,  0.4892],
                      [-0.4937, -0.2288],
                      [-0.3170, -0.1765]])),
             ('W_key',
              tensor([[-0.1833,  0.1258],
                      [ 0.2312,  0.0657],
                      [ 0.4019,  0.1622]])),
             ('W_value',
              tensor([[-0.0769,  0.4273],
                      [ 0.1779, -0.4363],
                      [ 0.2178,  0.1623]]))])

In [35]:
assert (sa_v1(inputs) == sa_v2(inputs)).sum(), "Error different weights"